In [1]:
import pandas as pd
import os
import tifffile
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import cv2


In [ ]:
# Default image size
default_size = (128, 128)

# Directory containing the images
image_dir = r'C:\Users\flopes1\OneDrive - Saint Louis University\Desktop\Repos\plant-disease-prediction\dataset\LARGO1\08-30'

# Find all image files starting with "imagem_segmentada_"
image_files = [filename for filename in os.listdir(image_dir) if filename.startswith('imagem_segmentada_')]

# List to store the channel data of all valid TIFF images
channel_data = []

# Load images, resize, and extract channel data
for filename in image_files:
    # Load image
    image_path = os.path.join(image_dir, filename)
    try:
        image = tifffile.imread(image_path)
        resized_image = cv2.resize(image, default_size)
        channel_data.extend(resized_image.transpose((2, 0, 1)))
    except tifffile.TiffFileError:
        print(f"Skipping file: {filename}. Not a valid TIFF file.")
    except cv2.error:
        print(f"Skipping file: {filename}. Error occurred during image resizing.")

# Stack channel data into a single array
channel_data = np.stack(channel_data, axis=-1)

# Calculate mean correlation matrix
mean_corr_matrix = np.corrcoef(channel_data.reshape(-1, channel_data.shape[-1]), rowvar=False)

# Create a heatmap chart for the correlation matrix
sns.heatmap(mean_corr_matrix, cmap='RdYlBu_r', annot=True, fmt='.2f')

# Display the chart
plt.title('Mean Correlation Matrix')
plt.show()

In [ ]:
import tifffile
import numpy as np
import os
import seaborn as sns
import matplotlib.pyplot as plt
import cv2


# Default image size
default_size = (128, 128)

# Directory containing the images
image_dirs = [
    r'C:\Users\flopes1\OneDrive - Saint Louis University\Desktop\Repos\plant-disease-prediction\dataset\LARGO1\08-30', 
    r'C:\Users\flopes1\OneDrive - Saint Louis University\Desktop\Repos\plant-disease-prediction\dataset\LARGO1\09-24', 
    r'C:\Users\flopes1\OneDrive - Saint Louis University\Desktop\Repos\plant-disease-prediction\dataset\LARGO1\10-05', 
    r'C:\Users\flopes1\OneDrive - Saint Louis University\Desktop\Repos\plant-disease-prediction\dataset\LARGO1\10-07', 
    r'C:\Users\flopes1\OneDrive - Saint Louis University\Desktop\Repos\plant-disease-prediction\dataset\LARGO1\10-26'
    ]

#image_dir = r'C:\Users\flopes1\OneDrive - Saint Louis University\Desktop\Repos\plant-disease-prediction\dataset\LARGO1\08-30'

# Initialize variables for covariance calculation
covariance_matrix = None
num_images = 0
# Iterate over each directory
for image_dir in image_dirs:
    # Find all image files starting with "imagem_segmentada_"
    image_files = [filename for filename in os.listdir(image_dir) if filename.startswith('imagem_segmentada_')]

    # Load and resize the images while calculating the covariance
    for image_file in image_files:
        image_path = os.path.join(image_dir, image_file)
        try:
            image = tifffile.imread(image_path)
            resized_image = cv2.resize(image, default_size)
            resized_image = resized_image.astype(np.float64)  # Convert to float64

            num_images += 1

            if covariance_matrix is None:
                # Reshape the resized image to a 2D array
                reshaped_image = resized_image.reshape(-1, resized_image.shape[-1])
                covariance_matrix = np.cov(reshaped_image, rowvar=False)
            else:
                reshaped_image = resized_image.reshape(-1, resized_image.shape[-1])
                covariance_matrix += np.cov(reshaped_image, rowvar=False)

        except tifffile.TiffFileError:
            print(f"Skipping file: {image_file}. Not a valid TIFF file.")
        except cv2.error:
            print(f"Skipping file: {image_file}. Error occurred during image resizing.")

# Divide the accumulated covariance matrix by the number of images to get the average
covariance_matrix /= len(image_dirs)
normalized_covariance_matrix = (covariance_matrix - np.min(covariance_matrix)) / (np.max(covariance_matrix) - np.min(covariance_matrix))

# Set Seaborn style
sns.set_style("ticks")
sns.set_context("paper")

# Create a heatmap chart for the correlation matrix
sns.heatmap(normalized_covariance_matrix, cmap='RdYlBu_r', annot=True, fmt='.2f')

# Set the x-axis and y-axis labels
channel_names = ["Blue", "Green", "Red", "Red Edge", "Near Infrared"]
plt.xticks(np.arange(len(channel_names)) + 0.5, channel_names, rotation=45)
plt.yticks(np.arange(len(channel_names)) + 0.5, channel_names, rotation=0)

# Display the chart
plt.title('Covariation Matrix')
plt.show()

In [ ]:
# Select a pixel or region of interest (ROI)
x = 50  # X-coordinate of the pixel/ROI
y = 50  # Y-coordinate of the pixel/ROI

# Extract the spectral profile at the selected pixel/ROI
spectral_profile = resized_image[y, x, :]

# Plot the spectral profile
wavelengths = np.arange(1, spectral_profile.shape[0] + 1)  # Assuming wavelengths start from 1
plt.plot(wavelengths, spectral_profile)
plt.xlabel('Wavelength')
plt.ylabel('Intensity')
plt.title('Spectral Profile')
plt.show()

In [7]:
!pip install gdal==3.6.4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 758.1/758.1 kB 9.0 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
  Created wheel for gdal: filename=GDAL-3.6.4-cp310-cp310-macosx_11_0_arm64.whl size=1072622 sha256=ee29e8d09c9f4d11049b3ff42e4ac1ef9acc35ff8017c28936c3dff93a0cc808
  Stored in directory: /Users/felipealencar/Library/Caches/pip/wheels/bb/f1/81/178bc46501f4dbf5fde926a079a2db035604ccef01d005e261
Successfully built gdal


In [2]:
!pip install dgl

  Obtaining dependency information for dgl from https://files.pythonhosted.org/packages/82/37/c230419e3c0da4f695bf15f682f7b8ec1cdefdb63dfd8f01aeadf5e93753/dgl-2.0.0-cp310-cp310-win_amd64.whl.metadata
  Obtaining dependency information for torchdata>=0.5.0 from https://files.pythonhosted.org/packages/0e/06/0c916f27ef9f5a566b555f07c82c94fb9277fcabe0fcbf4dfe4505dcb28a/torchdata-0.7.1-cp310-cp310-win_amd64.whl.metadata
  Obtaining dependency information for torch>=2 from https://files.pythonhosted.org/packages/7d/df/2c3f3a838b4fa5334ab79a5e0e4efacfb9a1a2fa1ab9e4d343be655fcb64/torch-2.2.1-cp310-cp310-win_amd64.whl.metadata
  Obtaining dependency information for typing-extensions>=4.8.0 from https://files.pythonhosted.org/packages/f9/de/dc04a3ea60b22624b51c703a84bbe0184abcd1d0b9bc8074b5d6b7ab90bb/typing_extensions-4.10.0-py3-none-any.whl.metadata
  Obtaining dependency information for sympy from https://files.pythonhosted.org/packages/d2/05/e6600db80270777c4a64238a98d442f0fd07cc8915be2a1c16d


[notice] A new release of pip is available: 23.2 -> 24.0
[notice] To update, run: C:\Users\flopes1\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [14]:
import importlib 
importlib.reload(utils) 
import utils as utils

file = r"C:\Users\flopes1\OneDrive - Saint Louis University\Desktop\Repos\plant-disease-prediction\dataset\LARGO2\08-30\CHACABUCO_LARGO_0830_ALL.tif"
shp_file = r"C:\Users\flopes1\OneDrive - Saint Louis University\Desktop\Repos\plant-disease-prediction\dataset\LARGO2\PLOT_BOUNDARIES\PLOTS_CHACABUCO_LARG_PROJECT.shp"
output_folder = r"C:\Users\flopes1\OneDrive - Saint Louis University\Desktop\Repos\plant-disease-prediction\dataset\LARGO2\08-30"

utils.split_plots(file, shp_file, output_folder, '0830', 'HEALTH_STA')

Shapefile opened successfully.


100%|██████████| 400/400 [00:24<00:00, 16.31it/s]
